# 01 — First look at the West Mercia crime data

This is an **independent** analysis of West Mercia (Herefordshire,
Shropshire, Telford and Wrekin, and Worcestershire), run the same way as
the London analysis in `notebooks/london/` but not assuming any of its
findings carry over. West Mercia is a much smaller, more rural force area
than the Metropolitan Police, so we check everything from scratch rather
than copy-pasting London's conclusions.

In [1]:
import sys
sys.path.append("../../src")

import pandas as pd
from load_data import load_force_data

wm = load_force_data("west-mercia")
wm.shape

(110499, 13)

**110,499 rows** — about 9% the size of London's 1.24M. Makes sense: West
Mercia covers roughly 1.3 million people across mostly rural/small-city
counties, versus Greater London's ~9 million in a dense urban area.

In [2]:
wm.isna().sum()

Crime ID                  18112
Month                         0
Reported by                   0
Falls within                  0
Longitude                  2099
Latitude                   2099
Location                      0
LSOA code                  2099
LSOA name                  2099
Crime type                    0
Last outcome category     18112
Context                  110499
source_file                   0
dtype: int64

### Finding 1: same ASB pattern as London

`Crime ID` is missing for 18,112 rows. Checked below — exactly like
London, this is 100% "Anti-social behaviour", the police's deliberate
anonymisation of ASB reports. Not a loading bug.

In [3]:
wm.loc[wm["Crime ID"].isna(), "Crime type"].value_counts()

Crime type
Anti-social behaviour    18112
Name: count, dtype: int64

### Finding 2: a pattern London did NOT have

2,099 rows are missing `Longitude`, `Latitude`, and `LSOA name` entirely —
London had zero rows like this. Checked what's actually in the `Location`
field for these rows below.

In [4]:
missing_geo = wm["LSOA name"].isna()
wm.loc[missing_geo, "Location"].value_counts()

Location
No Location    2099
Name: count, dtype: int64

In [5]:
wm.loc[missing_geo, "Crime type"].value_counts()

Crime type
Violence and sexual offences    1505
Anti-social behaviour            119
Other crime                       84
Public order                      78
Other theft                       77
Criminal damage and arson         61
Drugs                             47
Vehicle crime                     38
Shoplifting                       34
Burglary                          22
Possession of weapons             15
Robbery                           10
Theft from the person              7
Bicycle theft                      2
Name: count, dtype: int64

Every one of these 2,099 rows literally has `Location == "No Location"` —
a distinct data.police.uk value meaning no geography was published for
that crime at all (not even an anonymised point). Unlike the ASB pattern,
this spans every crime type, dominated by "Violence and sexual offences"
(~72%) but not exclusive to it. These rows are still valid for anything
that doesn't need geography (crime type counts, monthly trends), but must
be **excluded from any district-level analysis** — there's nothing to
attribute them to.

In [6]:
sorted(wm["Month"].unique())

['2025-05',
 '2025-06',
 '2025-07',
 '2025-08',
 '2025-09',
 '2025-10',
 '2025-11',
 '2025-12',
 '2026-01',
 '2026-02',
 '2026-03',
 '2026-04',
 '2026-05']

In [7]:
has_id = wm["Crime ID"].dropna()
has_id.duplicated().sum()

np.int64(37)

Same pattern as London's ~0.6% duplicate rate — repeated Crime IDs are a
known, real feature of this dataset (one incident filed under multiple
offence categories, or the same crime reappearing across monthly snapshots
with an updated outcome), not double-counted data.

In [8]:
wm["Crime type"].value_counts()

Crime type
Violence and sexual offences    43934
Anti-social behaviour           18112
Shoplifting                      9622
Criminal damage and arson        8382
Public order                     6944
Other theft                      6826
Burglary                         4412
Vehicle crime                    3577
Drugs                            2734
Other crime                      2623
Possession of weapons            1201
Robbery                           999
Bicycle theft                     785
Theft from the person             348
Name: count, dtype: int64

## Cleaning

Same redundant columns as London (`clean_crime_data` in `src/clean.py` —
reused as-is, since these are generic to the data.police.uk format, not
London-specific).

In [9]:
from clean import clean_crime_data

wm = clean_crime_data(wm)
wm.columns.tolist()

['Crime ID',
 'Month',
 'Longitude',
 'Latitude',
 'Location',
 'LSOA name',
 'Crime type',
 'Last outcome category']